# PARC2026 — 69b OpenVLA RLDS bridge smoke (recovery)

D10で選択した `V2_SQRT_BALANCED_RAW` の8 episodeだけを変換し、容量と入力形式を確認します。**下のコードセル1つを実行するだけです。** 依存インストール・変換・レポート確認・容量判定を順番に実行し、失敗した場合はその場で停止します。

既存のD10結果、Drive dataset、選択manifest、成功済みRLDS成果物は削除しません。検証済み環境と一致するsmokeレポートは再利用します。変換をやり直す場合も専用の新しいローカル作業ディレクトリを使います。

- CPU / L4 / A100対応。GPU学習・モデル重みのロード・全episode変換は行いません。
- Python 3.10の隔離環境を使用します。`promise==2.3` だけはTFDS互換のためソースビルドを許可し、PyAV 12.3.0を含む他の依存はbinary wheel必須のまま維持します。
- Colabに既存の古い `uv` があっても通る `--no-binary promise` 形式を使用し、wheel-only段階では `promise` を再要求しません。
- TensorFlow 2.17.1 / TFDS 4.9.6 と整合するよう、`tensorflow-metadata==1.16.1`、`protobuf==3.20.3`、`googleapis-common-protos==1.65.0` を固定します。新しいtensorflow-metadataとprotobuf<5の不整合を避けます。
- TFDS Builderをスクリプト直実行しても `__main__` をpackage resourceとして解決しないよう、converter側で `pkg_dir_path` を明示固定します。
- 失敗時は `bridge_smoke_status.json` と、そこに記録された実エラーログ末尾をNotebook自身が表示してから停止します。
- 完了後は `bridge_smoke_status.json` がPASSになり、変換レポートと容量判定を表示します。
- 8 episodeのsmokeでは `conversion_contract.json` は作りません。35GiBの閾値は次方式を選ぶための目安で、full変換やM3学習の開始許可ではありません。

**操作:** ランタイムを接続 → 下のセルを1回実行 → `=== 69b COMPLETE ===` を確認。以前の2/5・3/5・4/5セルや手動修正コマンドを追加実行する必要はありません。


In [ ]:
# 1セルで順番に復旧する。モデル学習・全量RLDS変換は行わない。
import json, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/parc2026')
ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / 'py_AI_69b_recovery'  # 既存checkoutと分離して安全に固定コミットを使う。
PIN = 'd2e2b2b213aa42bf84a5a90cb41f0e741658e42c'  # 69b script-safe TFDS builder + 回帰テスト。
URL = 'https://github.com/yu37330/py_AI.git'
if not (REPO / '.git').exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f'Existing non-Git directory: {REPO}. Please inspect it before continuing.')
    subprocess.run(['git', 'clone', '--no-checkout', URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', PIN], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', PIN], check=True)
got = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if got != PIN:
    raise RuntimeError(f'Repository pin mismatch: {got}')
print('69b recovery code:', got, flush=True)

OUT = Path('/content/drive/MyDrive/parc2026-cache/openvla-rlds-selected-v1')
cmd = [sys.executable, '-u', str(REPO / 'tools/colab/rlds_smoke_recovery.py'),
       '--root', str(ROOT), '--repo', str(REPO),
       '--drive', '/content/drive/MyDrive/parc2026-cache']

try:
    subprocess.run(cmd, check=True)
except subprocess.CalledProcessError:
    print('\n=== 69b DIAGNOSTICS ===', flush=True)
    status_path = OUT / 'bridge_smoke_status.json'
    status = None
    if status_path.is_file():
        try:
            status = json.loads(status_path.read_text(encoding='utf-8'))
            print('=== 69b STATUS ===', flush=True)
            print(json.dumps(status, ensure_ascii=False, indent=2), flush=True)
        except Exception as exc:
            print(f'Could not read status: {exc}', flush=True)
    else:
        print(f'Status file not found: {status_path}', flush=True)

    log_path = None
    error = status.get('error', '') if isinstance(status, dict) else ''
    if 'Full log: ' in error:
        log_path = Path(error.split('Full log: ', 1)[1].strip())
    logs = OUT / 'logs'
    if log_path is None or not log_path.is_file():
        candidates = sorted(logs.glob('*.log'), key=lambda p: p.stat().st_mtime, reverse=True) if logs.exists() else []
        log_path = candidates[0] if candidates else None
    if log_path and log_path.is_file():
        print('\n=== 69b ERROR LOG ===', flush=True)
        print('Log:', log_path, flush=True)
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print('\n'.join(lines[-200:]), flush=True)
    else:
        print('No error log found.', flush=True)
    raise

status = json.loads((OUT / 'bridge_smoke_status.json').read_text(encoding='utf-8'))
if status.get('status') != 'PASS':
    raise RuntimeError(f'69b did not complete: {status}')
print('=== 69b COMPLETE ===', flush=True)
print('Next: share the capacity decision; do not start notebook 70/M3 yet.', flush=True)
